# 技巧: 调试 `LinAlgError: Singular matrix` 错误

## 目的
在使用基于物理方程的数值模型（如`preissmann_model`）时，一个最常见的、也最让初学者困惑的错误就是 `numpy.linalg.LinAlgError: Singular matrix`。这个错误意味着模型在求解线性方程组 `M*dX = R` 时失败了。

本教程旨在解释这个错误背后的含义，并提供一些在水力模型中排查和解决此问题的常见思路和方法。

此笔记本将：
1.  解释“奇异矩阵（Singular matrix）”在数值求解中的意义。
2.  列举在`preissmann_model`中导致此错误的几种常见物理和数值原因。
3.  通过一个具体的例子，故意触发这个错误，并展示如何通过修改模型设置来解决它。

## 1. `LinAlgError` 是什么意思？

当`numpy.linalg.solve(M, R)`抛出`LinAlgError: Singular matrix`错误时，它在告诉你：你给我的系数矩阵`M`是“奇异的”或“不可逆的”。这意味着这个方程组要么**没有唯一解**，要么**有无穷多解**。求解器无法从中找出一个确定的解，因此只能报错退出。

在水力模型中，这几乎总是意味着你的模型在当前的物理状态下，进入了一个数学上无解或病态（ill-conditioned）的境地。

## 2. 导致错误的常见原因

1.  **河道干涸 (Dry Cells)**: **最常见的原因**。当河道中某个计算节点的水深`y`变得非常小、等于零甚至为负时，计算水力参数（如过水面积`A`、湿周`P`）的公式可能会出现除以零的错误，或者导致系数矩阵`M`中的某些行或列变为全零，从而使矩阵奇异。

2.  **超临界流 (Supercritical Flow)**: Preissmann格式是一个为缓流（Froude数 Fr < 1）设计的隐式格式。当水流流速过快，变成急流（Fr > 1）时，格式的稳定性和准确性会变差，可能导致解的振荡和最终的求解失败。

3.  **时间步长过大 (Large `dt`)**: 虽然隐式格式理论上对CFL条件不敏感，但过大的时间步长意味着在一个步长内物理状态的变化可能非常剧烈，超出了线性化假设的适用范围，导致迭代无法收敛，最终产生奇异矩阵。

4.  **不合理的参数或边界条件**: 例如，将曼宁糙率`n`设为0，或者给定一个与物理实际严重不符的边界条件，都可能导致模型进入一个无解的状态。

## 3. 触发并解决一个`LinAlgError`

我们来故意制造一个“河道干涸”的场景。我们建立一个模型，但将其初始水位设置得非常低，甚至低于河床的某些部分。

In [ ]:
import sys, os, time
import numpy as np
sys.path.insert(0, os.path.abspath(os.path.join(os.path.dirname('__file__'), '..')))

from preissmann_model.model import HydraulicModel
from preissmann_model.reach import RiverReach
from preissmann_model.cross_section import RectangularCrossSection

# --- 准备模型 ---
def create_model(initial_z, dt=10.0):
    reach = RiverReach(
        cross_sections=[RectangularCrossSection(width=10) for _ in range(5)],
        lengths=np.full(4, 200.0), slope=0.002, manning_n=0.03
    )
    model = HydraulicModel("debug_model", reach, dt, downstream_level=1.0)
    model.Q[:] = 1.0
    # Z_bed 的最高点在 Node 0, Z_bed[0] = 0.002 * 800 = 1.6
    # 我们将所有水位都设为一个固定值
    model.Z[:] = initial_z
    return model

# --- 场景一: 触发错误 ---
# 我们设置一个非常低的初始水位 (1.2m), 低于上游的河床高程 (1.6m)
print("--- 场景一: 初始水位过低 ---")
model_fail = create_model(initial_z=1.2)
try:
    model_fail.step({'Q_inflow': 1.0}, 10.0)
except np.linalg.LinAlgError as e:
    print(f"成功捕获到错误: {e}")
    print("原因: 初始水位低于上游河床高程，导致水深为负，矩阵奇异。")

# --- 场景二: 解决问题 ---
# 我们设置一个合理的初始水位 (2.0m), 确保所有节点都有水深
print("\n--- 场景二: 合理的初始水位 ---")
model_success = create_model(initial_z=2.0)
try:
    model_success.step({'Q_inflow': 1.0}, 10.0)
    print("模型运行一步成功!")
    print(f"新的出口流量: {model_success.get_outflow():.3f} m³/s")
except np.linalg.LinAlgError as e:
    print(f"出乎意料的错误: {e}")

## 4. 调试策略总结

当你遇到`LinAlgError`时，可以尝试以下调试策略：
1.  **检查初始条件**: 确保你的初始水位`Z`在所有节点上都高于河床高程`Z_bed`，保证初始水深`y = Z - Z_bed`处处为正。
2.  **减小时间步长 `dt`**: 这是最简单、最常用的“急救”措施。将`dt`减半或减小一个数量级，看看模型是否能稳定运行。如果可以，说明问题可能与数值稳定性有关。
3.  **检查边界条件**: 确保你提供的上、下游边界条件是物理上合理的。
4.  **打印中间状态**: 在你的代码中加入打印语句，在`step`被调用之前，打印出`model.Z`, `model.Q`和水深`model.Z - model.Z_bed`。这可以帮助你定位到是哪个节点、在哪个时刻首先出现了问题（例如，水深变为负值）。